# 02 — Silver transformation

Flag suspicious measurements and invalid geography/periods, enrich pickup zone, preserve all source records, then persist as Delta.

**Execution status:** Code is provided for reproducibility. See `results/observed_results.json` for results independently reported from an earlier Fabric run; this generated notebook has not been executed in this environment.

In [ ]:
from pyspark.sql import functions as F
BRONZE_PATH="Files/yellow_taxi/yellow_tripdata_2025-01.parquet"
LOOKUP_PATH="Files/reference/taxi_zone_lookup.csv"
# For local Spark, substitute data/raw/... and data/reference/...
df=spark.read.parquet(BRONZE_PATH)
zone_df=spark.read.option("header",True).option("inferSchema",True).csv(LOOKUP_PATH)
assert zone_df.select("LocationID").distinct().count()==zone_df.count(), "Duplicate lookup IDs"

In [ ]:
df_silver=(df
 .withColumn("geographic_quality_flag",F.when(F.col("PULocationID")==264,"UNKNOWN").when(F.col("PULocationID")==265,"OUTSIDE_NYC").otherwise("VALID"))
 .withColumn("reporting_period_flag",F.when(F.col("tpep_pickup_datetime").isNull(),"MISSING_TIMESTAMP").when(F.col("tpep_pickup_datetime")<F.lit("2025-01-01"),"BEFORE_PERIOD").when(F.col("tpep_pickup_datetime")>=F.lit("2025-02-01"),"AFTER_PERIOD").otherwise("IN_PERIOD"))
 .withColumn("distance_quality_flag",F.when(F.col("trip_distance").isNull()|F.isnan("trip_distance"),"MISSING_OR_NAN").when(F.col("trip_distance")<0,"NEGATIVE").when(F.col("trip_distance")==0,"ZERO").when(F.col("trip_distance")>100,"EXTREME").otherwise("NORMAL"))
 .withColumn("trip_duration_minutes",F.expr("timestampdiff(SECOND,tpep_pickup_datetime,tpep_dropoff_datetime)/60.0"))
 .withColumn("duration_quality_flag",F.when(F.col("trip_duration_minutes").isNull(),"MISSING").when(F.col("trip_duration_minutes")<0,"NEGATIVE").when(F.col("trip_duration_minutes")==0,"ZERO").when(F.col("trip_duration_minutes")>180,"LONG").otherwise("NORMAL"))
 .withColumn("pickup_eligible_flag",(F.col("PULocationID").isNotNull()) & (F.col("PULocationID")!=264) & (F.col("PULocationID")!=265) & (F.col("tpep_pickup_datetime")>=F.lit("2025-01-01")) & (F.col("tpep_pickup_datetime")<F.lit("2025-02-01")))
)
# Align eligibility to geographic quality even for unexpected PULocationID values.
df_silver=df_silver.withColumn("pickup_eligible_flag",(F.col("geographic_quality_flag")=="VALID")&(F.col("reporting_period_flag")=="IN_PERIOD"))
lookup=(zone_df.select(F.col("LocationID").alias("pickup_location_id"),F.col("Borough").alias("pickup_borough"),F.col("Zone").alias("pickup_zone"),F.col("service_zone").alias("pickup_service_zone")))
df_silver=df_silver.join(F.broadcast(lookup),df_silver.PULocationID==lookup.pickup_location_id,"left").drop("pickup_location_id")

In [ ]:
original=df.count(); preserved=df_silver.count(); eligible=df_silver.filter("pickup_eligible_flag").count(); missing=df_silver.filter(F.col("pickup_zone").isNull()).count()
print(dict(original=original,preserved=preserved,eligible=eligible,ineligible=preserved-eligible,missing_pickup_zones=missing))
assert original==preserved
assert eligible+(preserved-eligible)==preserved
assert missing==0
for c in ("geographic_quality_flag","reporting_period_flag","distance_quality_flag","duration_quality_flag"):
    print(c);df_silver.groupBy(c).count().show()

In [ ]:
TABLE="silver_yellow_taxi_trips"
if spark.catalog.tableExists(TABLE):
    print("Existing table unchanged:",TABLE)
else:
    df_silver.write.format("delta").mode("errorifexists").saveAsTable(TABLE)
print("Persisted records:",spark.table(TABLE).count())